# Проверка API T-ECD
Перед запуском обучите модель и запустите uvicorn из `fastaip/ml_service`.

In [ ]:
import json, random, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import requests
from tecd_pipeline import load_events

events = load_events("data/t_ecd", max_rows=100_000)
user_ids = events.user_id.dropna().astype(int).unique().tolist()
URL = "http://localhost:8079/predict"

In [ ]:
payload = {"user_id": random.choice(user_ids), "top_k": 5, "domain": "marketplace"}
response = requests.post(URL, json=payload, timeout=10)
response.raise_for_status()
response.json()

In [ ]:
def send(_):
    started = time.perf_counter()
    r = requests.post(URL, json={"user_id": random.choice(user_ids), "top_k": 5}, timeout=10)
    return {"status": r.status_code, "latency_ms": (time.perf_counter()-started)*1000}

with ThreadPoolExecutor(max_workers=10) as pool:
    results = pd.DataFrame(pool.map(send, range(100)))
results.describe(include="all")